# 01 — From main sequence to white dwarf

The goal is to evolve a single ZAMS (Zero-Age Main Sequence) star with  SeBa in AMUSE.

Chapter 5 of the AMUSE book is good to read for this


In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
from amuse.community.seba import Seba
from amuse.datamodel import Particle
from amuse.units import units


def locate_project_root():
    """Locate the project root independently of the notebook working directory. This is useful for running notebooks from whatever directory you want to work from."""
    cwd = Path.cwd().resolve()

    for parent in (cwd, *cwd.parents):
        for candidate in (parent, parent / "white_dwarf_project"):
            if (candidate / "notebooks" / "01_stellar_evolution.ipynb").is_file():
                return candidate

    raise FileNotFoundError(
        f"Cannot locate white_dwarf_project from {cwd}"
    )


ROOT = locate_project_root()
DATA_DIR = ROOT / "data"
FIGURE_DIR = ROOT / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)


## 1. Choose a progenitor star

A 1.2 solar-mass progenitor as an initial demonstration. We should eventually compare progenitor tracks and select those compatible with the star the micro-lensing event observed (I think 0.49 solar masses?).

In [ ]:
M_INITIAL = 1.2 | units.MSun
METALLICITY = 0.02
MAX_AGE = 14000 | units.Myr
MAX_STEPS = 60000

#Noting down here the AMUSE stellar_type IDs: 3 first giant branch, 5/6 AGB, 10/11/12 white dwarfs.
#You can check in the block below which stellar type the remnant is at the end of the evolution (Spoiler alert: It's 11, a CO white dwarf, as expected))
WHITE_DWARF_TYPES = {10, 11, 12}

#The numbers below will need to be adjusted if we change the stellar evolution code.
def next_output_step(stellar_type):
    """Pilot time sampling: finer output near giant and AGB phases since they evolve faster"""
    if stellar_type in (5, 6):
        return 0.01 | units.Myr  # AGB: 10 kyr
    if stellar_type in (2, 3):
        return 0.2 | units.Myr   # HG / RGB: 200 kyr
    if stellar_type == 4:
        return 0.5 | units.Myr   # Core He burning
    return 5 | units.Myr          # Main sequence


## 2. Run SeBa

We use `stellar.evolve_model(target_time)` and inspect the particle inside the SeBa worker. We stop the worker in a `finally` block so an interrupted calculation does not leave a process behind. The timestep used below determines when we save outputs that SeBa produces anyways, it handles its own timesteps internally.

In [ ]:
stellar = Seba()
records = []
start = perf_counter()

try:
    stellar.parameters.metallicity = METALLICITY
    star = stellar.particles.add_particle(Particle(mass=M_INITIAL))

    def save_current_state():
        records.append({
            "time_Myr": float(stellar.model_time.value_in(units.Myr)),
            "mass_MSun": float(star.mass.value_in(units.MSun)),
            "radius_RSun": float(star.radius.value_in(units.RSun)),
            "luminosity_LSun": float(star.luminosity.value_in(units.LSun)),
            "stellar_type": int(star.stellar_type.value_in(units.stellar_type)),
        })

    save_current_state()
    for _ in range(MAX_STEPS):
        kind = records[-1]["stellar_type"]
        if kind in WHITE_DWARF_TYPES or stellar.model_time >= MAX_AGE:
            break

        target = stellar.model_time + next_output_step(kind)
        if target > MAX_AGE:
            target = MAX_AGE

        stellar.evolve_model(target)
        save_current_state()
    else:
        raise RuntimeError("MAX_STEPS reached before WD formation or MAX_AGE.")
finally:
    stellar.stop()

track = {key: np.array([row[key] for row in records]) for key in records[0]}
print(f"Saved {len(records):,} stellar states.")
print(f"Elapsed wall time: {perf_counter()-start:.2f} s")
print(f"Final time: {track['time_Myr'][-1]/1000:.3f} Gyr")
print(f"Final mass: {track['mass_MSun'][-1]:.3f} Msun")
print(f"Final stellar type: {track['stellar_type'][-1]}")
print("White dwarf reached:", track["stellar_type"][-1] in WHITE_DWARF_TYPES)


## 3. Plot the evolution

One plot to show when the star increases its radius, when mass is lost, and when it becomes a compact remnant. The horizontal dashed line marks a 1-au radius around which we will probably place our planet.

In [ ]:
# I will note down here that the radius crossing 1 AU does not mean the planet is engulfed, since the star will have lost a lot of mass by then
#  and the planet will have moved outwards. But it is a good indicator of when the star is in its giant phases and when the planet is at risk of being engulfed.

time_Gyr = track["time_Myr"] / 1000
mass = track["mass_MSun"]
radius = track["radius_RSun"]
luminosity = track["luminosity_LSun"]

fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True, constrained_layout=True)

axes[0].plot(time_Gyr, mass)
axes[0].set_ylabel(r"Mass [$M_\odot$]")

axes[1].plot(time_Gyr, radius)
axes[1].axhline(
    (1 | units.au).value_in(units.RSun),
    linestyle="--", linewidth=1, color="gray", label="1 au",
)
axes[1].set_yscale("log")
axes[1].set_ylabel(r"Radius [$R_\odot$]")
axes[1].legend(frameon=False)

axes[2].plot(time_Gyr, luminosity)
axes[2].set_yscale("log")
axes[2].set_ylabel(r"Luminosity [$L_\odot$]")
axes[2].set_xlabel("Time since ZAMS [Gyr]")

for ax in axes:
    ax.grid(alpha=0.2)

fig.suptitle(f"SeBa stellar evolution: initial mass = {M_INITIAL.value_in(units.MSun):g} Msun")
output_figure = FIGURE_DIR / "stellar_evolution_pilot.pdf"
fig.savefig(output_figure, bbox_inches="tight")
plt.show()
print("Figure saved:", output_figure)


## 4. Diagnostics report

The maximal stellar radius is a useful diagnostic for engulfment. It is not validated until we refine the output cadence, especially near the AGB radius maximum.

In [ ]:
imax = int(np.argmax(radius))
print(f"Maximum sampled radius: {radius[imax]:.2f} Rsun")
print(f"                         {radius[imax] * (1 | units.RSun).value_in(units.au):.3f} au")
print(f"At time: {time_Gyr[imax]:.4f} Gyr")
print(f"Stellar type at max radius: {track['stellar_type'][imax]}")

wd_indices = np.flatnonzero(np.isin(track['stellar_type'], list(WHITE_DWARF_TYPES)))
if len(wd_indices):
    iwd = int(wd_indices[0])
    print(f"First sampled WD stage: {time_Gyr[iwd]:.4f} Gyr")
    print(f"White dwarf mass: {mass[iwd]:.4f} Msun")
else:
    print("No WD phase reached; try adjusting initial mass or MAX_AGE.")


## 5. Save data for followup

We store NumPy arrays for plots and analytical tests. Later, particle-based runs can use AMUSE's native HDF5 output.

In [ ]:
output_data = DATA_DIR / "stellar_track_pilot.npz"
np.savez_compressed(
    output_data,
    **track,
    initial_mass_MSun=M_INITIAL.value_in(units.MSun),
    metallicity=METALLICITY,
)
print("Saved stellar track:", output_data)


In [ ]:
import csv
import hashlib
import  json
import os
import platform
import sys
import uuid
import warnings
from contextlib import contextmanager
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

REFINEMENTS = (16, 32, 64)
WORKER_OPTIONS = {}  

# These are tolerances for this comparison, not uncertainties on the stellar model
BOUNDARY_TOL_AU = 0.001
WD_MASS_TOL_MSUN = 1e-4
RADIUS_REL_TOL = 0.01
PLANET_MASS_MSUN = float((1.9 | units.MEarth).value_in(units.MSun))
BOUNDARY_ECCENTRICITY = 0.0  # A circular test orbit for the engulfment test.
RSUN_IN_AU = float((1 | units.RSun).value_in(units.au))

# Read the schedule from our pilot function so there is only one place to change it.
BASE_STEPS_MYR = {
    kind: float(next_output_step(kind).value_in(units.Myr))
    for kind in range(7)
}
TRACK_COLUMNS = (
    "time_Myr", "mass_MSun", "radius_RSun", "luminosity_LSun", "stellar_type",
)


def write_json(path, values):
    Path(path).write_text(json.dumps(values, indent=2, allow_nan=False) + "\n")


def write_csv(path, rows):
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


@contextmanager
def worker_directory(path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


def installed_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not available"


sampling_id = datetime.now(timezone.utc).strftime("stellar_%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:6]
SAMPLING_DIR = DATA_DIR / "stellar_sampling_runs" / sampling_id
SAMPLING_FIGURES = FIGURE_DIR / "stellar_sampling_runs" / sampling_id
SAMPLING_DIR.mkdir(parents=True, exist_ok=False)
SAMPLING_FIGURES.mkdir(parents=True, exist_ok=False)

sampling_environment = {
    "python": sys.version, "python_executable": sys.executable,
    "platform": platform.platform(),
    "packages": {name: installed_version(name) for name in
                 ("amuse-framework", "amuse-seba", "numpy")},
    "worker_options": WORKER_OPTIONS,
}
write_json(SAMPLING_DIR / "environment.json", sampling_environment)
print("New tracks and results:", SAMPLING_DIR)
print("Refinement    MS step [Myr]    RGB step [yr]    AGB step [yr]")
for refinement in REFINEMENTS:
    print(f"{refinement:10d} {BASE_STEPS_MYR[1]/refinement:16.5f} "
          f"{1e6*BASE_STEPS_MYR[3]/refinement:16.2f} "
          f"{1e6*BASE_STEPS_MYR[5]/refinement:16.2f}")

## (I added this later) 6. Does the time sampling matter?

There is an interval where the star loses almost 10% of its mass between two saved states. We can repeat the evolution with smaller steps and see if results change.

Each run starts with a new ZAMS star and divides all the pilot's requested steps by that factor.

The pilot file above stays as it is. Every new track goes in its own folder with the same columns that later notebooks can then read.

This was made to work with notebook 04 if that matters for later work.

In [ ]:
import csv
import hashlib
import  json
import os
import platform
import sys
import uuid
import warnings
from contextlib import contextmanager
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

REFINEMENTS = (16, 32, 64)
WORKER_OPTIONS = {} 

# These are tolerances just for this comparison, not uncertainties on the stellar model
BOUNDARY_TOL_AU = 0.001
WD_MASS_TOL_MSUN = 1e-4
RADIUS_REL_TOL = 0.01
PLANET_MASS_MSUN = float((1.9 | units.MEarth).value_in(units.MSun))
BOUNDARY_ECCENTRICITY = 0.0  # A circular reference orbit 
RSUN_IN_AU = float((1 | units.RSun).value_in(units.au))

# Read the schedule from our pilot function
BASE_STEPS_MYR = {
    kind: float(next_output_step(kind).value_in(units.Myr))
    for kind in range(7)
}
TRACK_COLUMNS = (
    "time_Myr", "mass_MSun", "radius_RSun", "luminosity_LSun", "stellar_type",
)


def write_json(path, values):
    Path(path).write_text(json.dumps(values, indent=2, allow_nan=False) + "\n")


def write_csv(path, rows):
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


@contextmanager
def worker_directory(path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


def installed_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not available"


sampling_id = datetime.now(timezone.utc).strftime("stellar_%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:6]
SAMPLING_DIR = DATA_DIR / "stellar_sampling_runs" / sampling_id
SAMPLING_FIGURES = FIGURE_DIR / "stellar_sampling_runs" / sampling_id
SAMPLING_DIR.mkdir(parents=True, exist_ok=False)
SAMPLING_FIGURES.mkdir(parents=True, exist_ok=False)

sampling_environment = {
    "python": sys.version, "python_executable": sys.executable,
    "platform": platform.platform(),
    "packages": {name: installed_version(name) for name in
                 ("amuse-framework", "amuse-seba", "numpy")},
    "worker_options": WORKER_OPTIONS,
}
write_json(SAMPLING_DIR / "environment.json", sampling_environment)
print("New tracks and results:", SAMPLING_DIR)
print("Refinement    MS step [Myr]    RGB step [yr]    AGB step [yr]")
for refinement in REFINEMENTS:
    print(f"{refinement:10d} {BASE_STEPS_MYR[1]/refinement:16.5f} "
          f"{1e6*BASE_STEPS_MYR[3]/refinement:16.2f} "  
          f"{1e6*BASE_STEPS_MYR[5]/refinement:16.2f}")




### Run the finer tracks

We stop at the first saved WD state like in the pilot. The pilot's limit of 60,000 would probably be too small for these runs.

Completed runs in this batch can be reused if we rerun the cell hopefully, sometimes it doesnt work!

In [ ]:
def validate_stellar_track(values):
    arrays = [np.asarray(values[key]) for key in TRACK_COLUMNS]
    if arrays[0].ndim != 1 or len(arrays[0]) < 2 or any(a.shape != arrays[0].shape for a in arrays):
        raise ValueError("Stellar columns must be equal-length 1D arrays.")
    if not all(np.isfinite(a).all() for a in arrays):
        raise ValueError("Non-finite stellar output.")
    if arrays[0][0] != 0 or not np.all(np.diff(arrays[0]) > 0):
        raise ValueError("Require increasing times starting at ZAMS.")
    if np.any(arrays[1] <= 0) or np.any(arrays[2] <= 0):
        raise ValueError("Mass and radius must be positive.")
    if not np.equal(arrays[4], np.floor(arrays[4])).all():
        raise ValueError("Invalid stellar-type values.")
    if arrays[4][-1] not in WHITE_DWARF_TYPES:
        raise ValueError("This run did not reach a white dwarf.")
    if np.any(np.isin(arrays[4][:-1], list(WHITE_DWARF_TYPES))):
        raise ValueError("Expected a track ending at its first saved WD state.")


def pack_stellar_states(records, refinement):
    values = np.asarray(records)
    result = {key: values[:, j].copy() for j, key in enumerate(TRACK_COLUMNS)}
    result["stellar_type"] = result["stellar_type"].astype(np.int64)
    result.update(
        initial_mass_MSun=np.asarray(float(M_INITIAL.value_in(units.MSun))),
        metallicity=np.asarray(float(METALLICITY)),
        request_refinement=np.asarray(refinement),
    )
    return result


def run_refined_track(refinement):
    if refinement < 1 or int(refinement) != refinement:
        raise ValueError("Refinement must be a positive integer.")
    refinement = int(refinement)
    folder = SAMPLING_DIR / f"refinement_{refinement:03d}"
    path = folder / "stellar_track.npz"
    info_path = folder / "run.json"
    inputs = {
        "initial_mass_MSun": float(M_INITIAL.value_in(units.MSun)),
        "metallicity": float(METALLICITY), "refinement": refinement,
        "base_steps_Myr": {str(k): v for k, v in BASE_STEPS_MYR.items()},
        "max_age_Myr": float(MAX_AGE.value_in(units.Myr)),
        "max_steps": int(MAX_STEPS * refinement),
        "packages": sampling_environment["packages"],
        "worker_options": WORKER_OPTIONS,
    }
    if folder.exists():
        if path.is_file() and info_path.is_file():
            info = json.loads(info_path.read_text())
            if info.get("status") == "first_white_dwarf_reached" and info.get("inputs") == inputs:
                if hashlib.sha256(path.read_bytes()).hexdigest() != info["sha256"]:
                    raise ValueError(f"Saved track has changed: {path}")
                with np.load(path, allow_pickle=False) as saved:
                    result = {key: saved[key].copy() for key in saved.files}
                validate_stellar_track(result)
                print(f"Reusing completed refinement {refinement}.", flush=True)
                return result, info["wall_seconds"], path
        raise RuntimeError(f"{folder} contains an incomplete or different run. Run the setup cell for a new batch.")
    folder.mkdir()
    write_json(info_path, {"status": "started", "inputs": inputs})
    records, stellar = [], None
    start = last_report = perf_counter()
    print(f"Starting refinement {refinement} ...", flush=True)
    with worker_directory(folder):
        try:
            stellar = Seba(**WORKER_OPTIONS)
            stellar.parameters.metallicity = METALLICITY
            star = stellar.particles.add_particle(Particle(mass=M_INITIAL))

            def record_state():
                records.append((
                    float(stellar.model_time.value_in(units.Myr)),
                    float(star.mass.value_in(units.MSun)),
                    float(star.radius.value_in(units.RSun)),
                    float(star.luminosity.value_in(units.LSun)),
                    int(star.stellar_type.value_in(units.stellar_type)),
                ))

            record_state()
            for step in range(inputs["max_steps"]):
                age, _, _, _, kind = records[-1]
                if kind in WHITE_DWARF_TYPES:
                    break
                if age >= inputs["max_age_Myr"]:
                    raise RuntimeError("MAX_AGE reached without a white dwarf.")
                if kind not in BASE_STEPS_MYR:
                    raise ValueError(f"No requested step defined for stellar type {kind}.")
                target = min(age + BASE_STEPS_MYR[kind]/refinement, inputs["max_age_Myr"])
                stellar.evolve_model(target | units.Myr)
                record_state()
                if records[-1][0] <= age:
                    raise RuntimeError("SeBa did not advance its time.")
                if (step+1) % 20000 == 0 or perf_counter()-last_report >= 30:
                    print(f"  {len(records):,} states; age = {records[-1][0]/1000:.6f} Gyr; "
                          f"elapsed = {(perf_counter()-start)/60:.1f} min", flush=True)
                    last_report = perf_counter()
            if records[-1][-1] not in WHITE_DWARF_TYPES:
                raise RuntimeError("Step or age limit reached before WD formation.")
            result = pack_stellar_states(records, refinement)
            validate_stellar_track(result)
        except BaseException as exc:
            if records:
                np.savez_compressed(folder / "incomplete_track.npz", **pack_stellar_states(records, refinement))
            write_json(info_path, {"status": "failed", "inputs": inputs,
                                  "saved_states": len(records), "exception": repr(exc)})
            raise
        finally:
            if stellar is not None:
                try:
                    stellar.stop()
                except Exception as exc:
                    warnings.warn(f"Could not stop SeBa cleanly: {exc!r}")
    elapsed = perf_counter() - start
    np.savez_compressed(path, **result)
    write_json(info_path, {
        "status": "first_white_dwarf_reached", "inputs": inputs,
        "n_states": len(records), "wall_seconds": elapsed,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    })
    print(f"  Finished: {len(records):,} states in {elapsed:.1f} s", flush=True)
    return result, elapsed, path

In [ ]:
# This is expected to take a while, for me it took about 20 minutes for the three refinements.

if len(REFINEMENTS) < 3 or any(int(r) != r or r < 1 for r in REFINEMENTS):
    raise ValueError("Choose at least three positive integer refinements.")
if any(b != 2*a for a, b in zip(REFINEMENTS[:-1], REFINEMENTS[1:])):
    raise ValueError("Use successive doublings, e.g. (16, 32, 64).")

write_json(SAMPLING_DIR / "settings.json", {
    "initial_mass_MSun": float(M_INITIAL.value_in(units.MSun)),
    "metallicity": float(METALLICITY), "refinements": list(REFINEMENTS),
    "base_steps_Myr": BASE_STEPS_MYR, "planet_mass_MSun": PLANET_MASS_MSUN,
    "boundary_eccentricity": BOUNDARY_ECCENTRICITY,
    "boundary_tolerance_AU": BOUNDARY_TOL_AU,
    "WD_mass_tolerance_MSun": WD_MASS_TOL_MSUN,
    "radius_relative_tolerance": RADIUS_REL_TOL,
})
refined_tracks, refinement_times, refinement_paths = {}, {}, {}
for refinement in REFINEMENTS:
    result, elapsed, path = run_refined_track(refinement)
    refined_tracks[refinement] = result
    refinement_times[refinement] = elapsed
    refinement_paths[refinement] = path

## 7. Comparing tracks

We compare the RGB and AGB separately, since a stable AGB maximum could hide a changing RGB result. We also check the WD mass and note down how much mass is lost between adjacent states.

For the geometric engulfment estimate, for now, we assume a slow, isotropic mass loss, constant eccentricity and a planet treated as a point particle. Then

$$a(t)=a_0\frac{M_{\star,0}+m_p}{M_\star(t)+m_p}, \qquad q(t)=a(t)(1-e_0),$$

so the initial semimajor axis needed to keep periapsis (the closest point in the orbit to the orbiting body, I learnt this researching for this) outside the star is

$$a_{0,\mathrm{crit}}=\max_t\left[ R_\star(t)\frac{M_\star(t)+m_p}{(M_{\star,0}+m_p)(1-e_0)}\right].$$

For now this uses the saved states, without too much complexity. It is a useful quantity to compare between tracks, but it is, of course, not our final survival boundary.

In [ ]:
def summarize_stellar_track(values, label, refinement=None, elapsed=None):
    validate_stellar_track(values)
    age = values["time_Myr"]
    mass = values["mass_MSun"]
    kind = values["stellar_type"]
    radius_AU = values["radius_RSun"] * RSUN_IN_AU
    if not 0 <= BOUNDARY_ECCENTRICITY < 1:
        raise ValueError("The initial eccentricity must be between 0 and 1.")
    critical = radius_AU * (mass + PLANET_MASS_MSUN) / (
        (mass[0] + PLANET_MASS_MSUN) * (1 - BOUNDARY_ECCENTRICITY)
    )
    imax = int(np.argmax(critical))
    fraction = abs(np.diff(mass)) / mass[:-1]
    imass = int(np.argmax(fraction))

    def phase_max(array, types):
        selected = np.isin(kind, types)
        return float(array[selected].max()) if selected.any() else None

    return {
        "run": label, "refinement": refinement, "n_states": len(age), "wall_seconds": elapsed,
        "first_WD_time_Myr": float(age[-1]),
        "WD_transition_bracket_yr": float(1e6*(age[-1]-age[-2])),
        "mass_WD_MSun": float(mass[-1]),
        "max_radius_RGB_AU": phase_max(radius_AU, [3]),
        "max_radius_AGB_AU": phase_max(radius_AU, [5, 6]),
        "a_critical_RGB_AU": phase_max(critical, [3]),
        "a_critical_AGB_AU": phase_max(critical, [5, 6]),
        "a_critical_all_AU": float(critical[imax]),
        "critical_time_Myr": float(age[imax]), "critical_stellar_type": int(kind[imax]),
        "max_adjacent_mass_fraction": float(fraction[imass]),
        "largest_mass_step_start_Myr": float(age[imass]),
        "largest_mass_step_end_Myr": float(age[imass+1]),
    }


sampling_summary = []
pilot_path = DATA_DIR / "stellar_track_pilot.npz"
if pilot_path.is_file():
    with np.load(pilot_path, allow_pickle=False) as saved:
        pilot_track = {key: saved[key].copy() for key in saved.files}
    same_inputs = (
        "initial_mass_MSun" in pilot_track and "metallicity" in pilot_track
        and np.isclose(float(pilot_track["initial_mass_MSun"]),
                       float(M_INITIAL.value_in(units.MSun)), rtol=1e-12, atol=0)
        and np.isclose(float(pilot_track["metallicity"]), METALLICITY, rtol=1e-12, atol=0)
    )
    if same_inputs:
        # The old pilot is a reference here; it does not count towards a convergence pass.
        sampling_summary.append(summarize_stellar_track(pilot_track, "original_pilot"))
    else:
        print("The saved pilot has different or missing input settings; leaving it out of this comparison.")

for refinement in REFINEMENTS:
    sampling_summary.append(summarize_stellar_track(
        refined_tracks[refinement], f"refinement_{refinement}",
        refinement, refinement_times[refinement],
    ))
write_csv(SAMPLING_DIR / "stellar_convergence_summary.csv", sampling_summary)

print(f"{'Run':<18} {'States':>8} {'M_WD':>10} {'R_RGB':>10} {'R_AGB':>10} "
      f"{'acrit_RGB':>11} {'acrit_AGB':>11} {'max dM/M':>11}")
for row in sampling_summary:
    show = lambda key: f"{row[key]:.6f}" if row[key] is not None else "missing"
    print(f"{row['run']:<18} {row['n_states']:8d} {show('mass_WD_MSun'):>10} "
          f"{show('max_radius_RGB_AU'):>10} {show('max_radius_AGB_AU'):>10} "
          f"{show('a_critical_RGB_AU'):>11} {show('a_critical_AGB_AU'):>11} "
          f"{100*row['max_adjacent_mass_fraction']:10.4f}%")
print("Masses are in Msun; radii and initial critical distances are in au.")

### Have the results settled?

We check for changes below 0.001 au in the geometric boundaries, 1% in the RGB and AGB maximum radii, and 0.0001 solar masses in the WD mass. Comparisons must pass all these checks.

I don't think we should take a smaller value as confirmation that the track has converged. We are not checking WD cooling or luminosity convergence here.

In [ ]:
def compare_stellar_summaries(coarse, fine):
    row = {"coarse": coarse["refinement"], "fine": fine["refinement"]}
    boundary_ok = radius_ok = True
    for key in ("a_critical_all_AU", "a_critical_RGB_AU", "a_critical_AGB_AU"):
        delta = abs(fine[key]-coarse[key]) if coarse[key] is not None and fine[key] is not None else None
        row["delta_" + key] = delta
        boundary_ok = boundary_ok and delta is not None and delta <= BOUNDARY_TOL_AU
    for key in ("max_radius_RGB_AU", "max_radius_AGB_AU"):
        delta = abs(fine[key]-coarse[key])/fine[key] if coarse[key] is not None and fine[key] not in (None, 0) else None
        row["relative_delta_" + key] = delta
        radius_ok = radius_ok and delta is not None and delta <= RADIUS_REL_TOL
    row["delta_mass_WD_MSun"] = abs(fine["mass_WD_MSun"]-coarse["mass_WD_MSun"])
    row["delta_first_WD_time_yr"] = 1e6*abs(fine["first_WD_time_Myr"]-coarse["first_WD_time_Myr"])
    row["boundaries_pass"] = bool(boundary_ok)
    row["radii_pass"] = bool(radius_ok)
    row["WD_mass_pass"] = row["delta_mass_WD_MSun"] <= WD_MASS_TOL_MSUN
    row["pass_working_tolerances"] = all(row[key] for key in ("boundaries_pass", "radii_pass", "WD_mass_pass"))
    return row


by_refinement = {row["refinement"]: row for row in sampling_summary if row["refinement"] is not None}
sampling_comparisons = [
    compare_stellar_summaries(by_refinement[coarse], by_refinement[fine])
    for coarse, fine in zip(REFINEMENTS[:-1], REFINEMENTS[1:])
]
write_csv(SAMPLING_DIR / "stellar_refinement_comparisons.csv", sampling_comparisons)
converged_for_test = len(sampling_comparisons) >= 2 and all(
    row["pass_working_tolerances"] for row in sampling_comparisons[-2:]
)
for row in sampling_comparisons:
    print(f"{row['coarse']} -> {row['fine']}: boundaries={row['boundaries_pass']}, "
          f"radii={row['radii_pass']}, WD mass={row['WD_mass_pass']}")
print("Overall:", "PASS for these working tolerances" if converged_for_test else "REVIEW / REFINE FURTHER")

In [ ]:
# Plot the stuff from the above comparison.
levels = np.array(REFINEMENTS)
fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True, constrained_layout=True)
for phase in ("RGB", "AGB"):
    axes[0].plot(levels, [by_refinement[r][f"max_radius_{phase}_AU"] for r in levels], "o-", label=phase)
    axes[1].plot(levels, [by_refinement[r][f"a_critical_{phase}_AU"] for r in levels], "o-", label=phase)
axes[2].plot(levels, [by_refinement[r]["mass_WD_MSun"] for r in levels], "o-")
axes[0].set_ylabel("Maximum sampled radius [au]")
axes[1].set_ylabel("Critical initial semimajor axis [au]")
axes[2].set_ylabel(r"WD mass [$M_\odot$]")
axes[2].set_xlabel("Refinement factor")
axes[2].set_xscale("log", base=2)
axes[2].set_xticks(levels, [str(r) for r in levels])
for ax in axes:
    ax.grid(alpha=0.2)
for ax in axes[:2]:
    ax.legend(frameon=False)
fig.suptitle("SeBa requested-step comparison")
fig.savefig(SAMPLING_FIGURES / "stellar_sampling_convergence.pdf", bbox_inches="tight")
plt.show()

## 8. Which track do we use next?

Each refinement above saves a `stellar_track.npz` that later notebooks (specifically made for notebook 04 once again) can read. The cell below records the result and prints the exact path to the finest track. We only call it selected if the last two comparisons passed.

If the result is still `REVIEW`, we can add maybe add more refinements?

In [ ]:
finest = max(REFINEMENTS)
finest_path = refinement_paths[finest]
sampling_status = {
    "status": "passed_working_tolerances" if converged_for_test else "needs_further_refinement",
    "passed_working_tolerances": bool(converged_for_test),
    "finest_refinement": int(finest),
    "finest_candidate_track": str(finest_path.relative_to(ROOT)),
    "selected_track": str(finest_path.relative_to(ROOT)) if converged_for_test else None,
    "sha256": hashlib.sha256(finest_path.read_bytes()).hexdigest(),
    "last_two_comparisons": sampling_comparisons[-2:],
    "finest_summary": by_refinement[finest],
    "scope": "Requested-step convergence of the first-WD mass and sampled radii/geometric boundaries.",
    "not_proof_of_physical_accuracy": True,
}
write_json(SAMPLING_DIR / "validation_status.json", sampling_status)
print("Selected track:" if converged_for_test else "Finest candidate (still provisional):", finest_path)
if converged_for_test:
    print("\n Use this path (We really need to make this automatic at some point when we turn all of this into a script.):")
    print(f'SEBA_TRACK_PATH = ROOT / "{finest_path.relative_to(ROOT).as_posix()}"')
else:
    print("The finest file is saved, but has not passed the convergence checks.")
print("\nFiles to inspect:")
for name in ("stellar_convergence_summary.csv", "stellar_refinement_comparisons.csv", "validation_status.json"):
    print(SAMPLING_DIR / name)

### Stuff we still need to do here

1. Refine the giant-phase sampling and verify the maximum radius converges.
2. Check other initial masses and identify tracks reaching a WD mass near (probably) 0.49 Msun.
3. Consider SeBa versus SSE differences. Is MESA an option?